<center>
    <img src="https://upload.wikimedia.org/wikipedia/commons/a/a8/%D0%9B%D0%9E%D0%93%D0%9E_%D0%A8%D0%90%D0%94.png" width=500px/>
    <font>Python 2025</font><br/>
    <br/>
    <br/>
    <b style="font-size: 2em">Async programming</b><br/>
    <br/>
    <font>Борис Плетенев, по материалам Ярослав Золотарёва🫡</font><br/>
</center>

IO-bound операции — операции, связанные с длительным ожиданием другого устройства, например, сетевой карты или диска


In [ ]:
with open("some_data.txt", "r", encoding="utf-8") as f:
    # blocks until all file will be read
    text = f.read()

print(text)

In [ ]:
import pymongo  # for async, use motor! 

client = MongoClient('localhost', 27017)

posts = client['web_db']['posts']

# blocks until DB answers
results = posts.find({'author': 'Undefined', 'date': date.today})

Запускаем гуся в чатик


In [ ]:
import requests

# blocks until site returns response
response = requests.get('https://ya.ru')

<center>
<img src="https://blog-assets.risingstack.com/2016/Apr/non_async_blocking_operations_example_in_node_hero-1459856858194.png" alt="io-operations" width=800/>
</center>

## Asynchronous I/O

<center>
<img 
src="https://www.koyeb.com/_next/image?url=%2Fstatic%2Fimages%2Fblog%2Fsync-vs-async-schema.png&w=750&q=75" alt="sync-vs-async" width=700/>
</center>

Asynchronous I/O — неблокирующая обработка ввода/вывода, которая позволяет процессу продолжить выполнение не дожидаясь окончания передачи данных.

# Корутины и Фьючи (coroutines and futures)

### с самого начала...

In [4]:
def eager_range(up_to: int) -> list[int]:
    sequence = []
    index = 0
    while index < up_to:
        sequence.append(index)
        index += 1
    return sequence

Основная идея - научиться как-то переключаться между задачами! При этом, не потеряв ее и завершив когда-то потом, когда будет время (или когда придут данные с I/O операции)

Начиная с Python 2.2 в языке появились генераторы

In [5]:
from collections.abc import Generator

In [6]:
def lazy_range(up_to: int) -> Generator[int, None, None]:
    index = 0
    while index < up_to:
        yield index
        index += 1

В Python 2.5 вводят метод `send()`

In [7]:
def jumping_range(up_to: int) -> Generator[int, int | None, None]:
    index = 0
    while index < up_to:
        jump = yield index
        if jump is None:
            jump = 1
        index += jump

In [8]:
generator = jumping_range(5)

In [9]:
print('next   :', next(generator))
print('send  2:', generator.send(2))
print('next   :', next(generator))
print('send -1:', generator.send(-1))

next   : 0
send  2: 2
next   : 3
send -1: 2


В Python 3.3 добавляется важный синтаксический сахар `yield from`

In [10]:
def bottom() -> Generator[int, None, int]:
    yield 42
    yield 84
    return 0

In [11]:
def top() -> Generator[int, None, None]:
    value = yield from bottom()
    print(value)

In [57]:
top()

list(top())

0


[42, 84]

Наконец, в Python 3.4 вводят фреймворк `asyncio`

In [13]:
import asyncio
import types

In [14]:
@types.coroutine  # asyncio.coroutine doesn't work after 3.10
def countdown(n: int) -> Generator[asyncio.Future, None, None]:
    while n > 0:
        print(n)
        yield from asyncio.sleep(1)
        n -= 1

In [15]:
next(countdown(3))

3


<Future pending>

In [58]:
loop = asyncio.get_event_loop()
loop.run_until_complete(countdown(3))

3
2
1


Довольно быстро становится понятно, что в языке теперь есть некоторая путаница между генераторами и корутинами

Поэтому в следующем же обновлении (Python 3.5) вводят `async/await`, заменив generator-based корутины на встроенные в язык

In [18]:
async def compute(a: int, b: int) -> int:
    print('Compute...')
    await asyncio.sleep(1)
    return a + b

In [19]:
compute(3, 5)

<coroutine object compute at 0x7e8913ad4930>

In [59]:
next(compute(3, 5).__await__())  # awaitable

Compute...


<Future pending>

In [80]:
loop = asyncio.get_event_loop()
res = loop.run_until_complete(compute(3, 5))
print(res)

Compute...
8


Future говорит: «когда-нибудь здесь будет значение или ошибка».

Coroutine говорит: «я готова подождать future и продолжить, когда оно будет готово» (await)

Future + Coroutine = Task

In [ ]:
import inspect
import asyncio

async def coroutine():
    return 42

async def main():
    # coroutine — just an object which is not executing!
    c = coroutine()
    print("c =", c)
    print("type(c) =", type(c))

    # is method __await__() supported:
    print("isawaitable(c) =", inspect.isawaitable(c)) 

    # Task — running coroutine + Future:

    # Making Task
    task = asyncio.create_task(c)
    print("task =", task)
    print("type(task) =", type(task))
    print("isinstance(task, asyncio.Future) =", isinstance(task, asyncio.Future))

    result = await task
    print(result)

# asyncio.run(main())
await main()

c = <coroutine object coroutine at 0x7e8912f0be20>
type(c) = <class 'coroutine'>
isawaitable(c) = True
task = <Task pending name='Task-71' coro=<coroutine() running at /tmp/ipykernel_78147/3654646180.py:4>>
type(task) = <class 'asyncio.tasks.Task'>
isinstance(task, asyncio.Future) = True
42


А затем в Python 3.6 появится возможность реализовывать асинхронные генераторы

In [23]:
async def ticker(upto: int) -> Generator[int, None, None]:
    for i in range(upto):
        print(f"iteration {i}")
        await asyncio.sleep(1)
        yield i

In [24]:
next(ticker(2).__anext__().__await__())

iteration 0


<Future pending>

In [25]:
loop = asyncio.get_event_loop()
loop.run_until_complete(ticker(2).__anext__())  # anext(...) from 3.10

iteration 0


0

# Event Loop

In [82]:
async def compute(a: int, b: int) -> int:
    print('Compute...')
    await asyncio.sleep(1)
    return a + b

In [83]:
async def print_sum(a: int, b: int) -> None:
    result = await compute(a, b)
    # result = await compute(a, b)
    print(f'{a} + {b} = {result}')

In [ ]:
loop = asyncio.get_event_loop()
loop.run_until_complete(print_sum(1, 2))

loop.run_until_complete(print_sum(3, 4))

Compute...
1 + 2 = 3
Compute...
3 + 4 = 7


<center>
<img src="http://ntoll.org/static/images/tulip_coro.png" alt="event-loop" width=1200/>
</center>

У event loop’а есть такие сущности:

1. Очередь готовых callbacks/тасок — то, что надо выполнить прямо сейчас.(deque)

2. Очередь таймеров (delayed calls) — это приоритетная очередь по времени исполнения (heapq).

3. Регистрация I/O-событий — обёртка вокруг select()/epoll()/kqueue() через модуль selectors. (from select import select)

4. Signal handlers / callbacks из других потоков — тоже в итоге сводятся к push в очередь готовых callback’ов. 
(loop.call_soon_threadsafe(cb, *args))

# Современный asyncio

## python 3.9+

## Hello world

In [31]:
async def main() -> None:
    print('Hello ...')
    await asyncio.sleep(1)
    print('... World!')

In [ ]:
asyncio.run(main())

# or just await if you are in ipynb file

await main()

Hello ...
... World!


## Шедулинг корутин

In [85]:
async def say_after(delay: int, what: str) -> None:
    print(f'wait for {delay}s')
    await asyncio.sleep(delay)
    print(what)

In [86]:
import time

In [87]:
print(f"started at {time.strftime('%X')}")

await say_after(1, 'hello')
# say_after(2, 'world')
await say_after(2, 'world')

print(f"finished at {time.strftime('%X')}")

started at 18:34:46
wait for 1s
hello
wait for 2s
world
finished at 18:34:49


In [92]:
task1 = asyncio.create_task(say_after(2, 'hello'))
task2 = asyncio.create_task(say_after(1, 'world'))

print(f"started at {time.strftime('%X')}")

# await task1
# await task2

print(f"finished at {time.strftime('%X')}")

started at 18:43:15
finished at 18:43:15


wait for 2s
wait for 1s
world
hello


Задачи (tasks) используются, чтобы запланировать (schedule) корутины на выполнение "параллельно"

Когда из корутины создают задачу через `asyncio.create_task()`, она автоматически запускается на следующем такте event loop'а (run soon)

## asyncio.gather

In [37]:
async def factorial(name: str, number: int) -> None:
    result = 1
    for i in range(2, number + 1):
#         if number % 2 and i == 2:
#             raise RuntimeError('Whoops!')
        print(f'Task {name}: Compute factorial({i})...')
        await asyncio.sleep(1)
        result *= i
    print(f'Task {name}: factorial({number}) = {result}')
    return result

In [38]:
await asyncio.gather(
    factorial('A', 2),
    factorial('B', 3),
    factorial('C', 4),
#     return_exceptions=True,
)

# Q: How to parse exceptions? If there are 2 different exceptions? 
# A: ExceptionGroup since 3.11

Task A: Compute factorial(2)...
Task B: Compute factorial(2)...
Task C: Compute factorial(2)...
Task A: factorial(2) = 2
Task B: Compute factorial(3)...
Task C: Compute factorial(3)...
Task B: factorial(3) = 6
Task C: Compute factorial(4)...
Task C: factorial(4) = 24


[2, 6, 24]

## asyncio.TaskGroup
#### Python 3.11: structured concurrency, ExceptionGroup

In [64]:
async def factorial(name: str, number: int) -> None:
    result = 1
    for i in range(2, number + 1):
        if number == 2:
            raise RuntimeError('Whoops 2!')
        if number == 3:
            raise ValueError('Whoops 3!')
        print(f'Task {name}: Compute factorial({i})...')
        await asyncio.sleep(1)
        result *= i
    print(f'Task {name}: factorial({number}) = {result}')
    return result

In [65]:
try:
    async with asyncio.TaskGroup() as tg:
        tg.create_task(factorial('A', 2))
        tg.create_task(factorial('B', 3))
        tg.create_task(factorial('C', 4))
        print('tasks pending to start')
except* (ValueError, RuntimeError) as excgroup: 
    # * in except mostly important for coroutines! Before that, we have to catch all exceptions by hand!
    for exc in excgroup.exceptions:
        print(f'Caught exception: {exc} with class {exc.__class__.__name__}')

print('tasks complete')

tasks pending to start
Task C: Compute factorial(2)...
Caught exception: Whoops 2! with class RuntimeError
Caught exception: Whoops 3! with class ValueError
tasks complete


## Ожидание & таймауты

In [ ]:
async def eternity() -> None:
    await asyncio.sleep(3600)  # 1 hour sleep
    print('yay!')

In [96]:
try:
    await asyncio.wait_for(eternity(), timeout=1)  # wait for at most 1 second
except TimeoutError:
    print('timeout!')

timeout!


## Отмена (cancellation) корутин

In [97]:
async def another_eternity() -> None:
    try:
        await asyncio.sleep(3600)
        print(f'{time.strftime("%X")} yay!')
    except asyncio.CancelledError:
        print(f"{time.strftime("%X")} Hey, I am cancelled!")
        raise
    finally:
        await asyncio.sleep(3)  # 30
        print(f'{time.strftime("%X")} coro finished')

In [98]:
print(f'{time.strftime("%X")} started')
try:
    # Send CancelledError to coro after 1 sec and wait for coro completion
    await asyncio.wait_for(another_eternity(), timeout=1)
except asyncio.TimeoutError:
    print(f'{time.strftime("%X")} timeout!')
    
print(f'{time.strftime("%X")} block finished')

18:54:29 started
18:54:30 Hey, I am cancelled!
18:54:33 coro finished
18:54:33 timeout!
18:54:33 block finished


## Защита (shield) от отмены корутин

In [101]:
async def important_task() -> None:
    try:
        print(f'{time.strftime("%X")} important task started')
        await asyncio.sleep(10)
        print(f'{time.strftime("%X")} important task finished')
    except asyncio.CancelledError:
        print(f"{time.strftime("%X")} Hey, I am cancelled!")
        raise

In [102]:
print(f'{time.strftime("%X")} started')
try:
    await asyncio.wait_for(asyncio.shield(important_task()), timeout=1.0)
except asyncio.TimeoutError:
    print(f'{time.strftime("%X")} timeout!')
    
print(f'{time.strftime("%X")} block finished')

18:56:01 started
18:56:01 important task started
18:56:02 timeout!
18:56:02 block finished


18:56:11 important task finished


## Python 3.13: as_completed

In [47]:
async def factorial(number: int) -> tuple[int, int]:
    result = 1
    for i in range(2, number + 1):
        await asyncio.sleep(1)
        result *= i
    return number, result

In [48]:
for i, future in enumerate(asyncio.as_completed([factorial(4), factorial(3),
                                                 factorial(5), factorial(2)])):
    number, result = await future
    print(f"Factorial({number}) = {result}")

Factorial(2) = 2
Factorial(3) = 6
Factorial(4) = 24
Factorial(5) = 120


## async with

Асинхронный контекстный менеджер - это контекстный менеджер,

который умеет приостанавливать выполнение в методах

входа и выхода: `__aenter__()`, `__aexit__()`

In [ ]:
lock = asyncio.Lock()

async with lock as l:
    # access shared state
    # use l as you want!

    # Coro 1
    await asyncio.sleep(1)
    # Coro 2
    await asyncio.sleep(2)

### Free-threaded Python!

От GIL можно избавиться! \
А значит, у нас на вооружении есть целый зоопарк. Вы только представьте, один процесс, много тредов, каждый имеет свой ивент луп, каждый обрабатывает асинхронно запросы! Сколько перфа!

### aiohttp

In [78]:
import aiohttp

async with aiohttp.ClientSession() as session:  # closing connection requires having an active event loop
    async with session.get('http://ya.ru') as resp:
        text = await resp.text()
        print(text[:70], '...')

<!doctype html><html lang="ru" class="Theme Theme_color_morda-default  ...


        Задача A (fetch)            Event loop                ОС + NIC              Другая задача B
--------------------------------------------------------------------------------------------------------
1) start fetch()
   async with session.get(...)
   |
   |  вызываем aiohttp: 
   |  создаётся корутина 
   |   для HTTP-запроса,
   |  внутри открывается
   |  неблокирующий сокет
   |--------------------->  регистрирует сокет (FD) 
                            в epoll/select
                            делает connect()/send()
                            через системные вызовы
                            отправляет HTTP-запрос
                            ------------------------>  пакеты уходят по сети
                                                    (карточка → интернет → сервер)

2) await session.get(...)
   |
   |  доходит до первого await 
   | "ждать, пока сокет 
   |   станет readable"
   |--------------------->  event loop видит await на future,
                            помечает задачу A как "ожидающую I/O"
                            НЕ блокирует поток, а уходит в selector.select()
                            или берёт следующую готовую задачу
                            -------------------------------------------------->  переходит к задаче B

3) Задача A "спит"         Event loop сейчас крутит B
   (она вообще не          (или ещё кого-то: C, D, таймеры,
   занимает CPU)           периодические задачи и т.п.)
   |                       |                                |
   |                       |                                |
   |                       |                                |
   |                       |                                |
   |                       |                                |   Сервер шлёт ответ
   |                       |                                |<------------------------
   |                       |                                |  Пакеты приходят на карточку

4) Ответ приходит          Event loop всё ещё в select()/выполняет B
                           |                                |
                           |                                <-- карточку получает пакеты
                           |                                    генерирует hardware interrupt
                           |                                <-- ядро начинеает его обрабатывать
                           |                                <-- раскладывает данные по TCP-стеку
                           |                                <-- кладёт байты в приёмный буфер сокета
                           |                                <-- помечает "FD ready for read"

5) Ядро будит select       Event loop
   (epoll/select возвращает "сокет готов")
                           |
                           |<------------------------------ epoll/select возвращается
                           |   "FD readable"
                           |
                           |   event loop находит future,
                           |   связанное с этим сокетом,
                           |   отмечает его как выполненный
                           |   и ставит задачу A в очередь ready

6) Свич с B на A            Event loop
   (B suspended or done)   |
   |                       |   заканчивает текущий тик для B
   |                       |   берёт из очереди готовых задач A
   |<----------------------|
   |   задача A продолжается
   |   после await session.get(...)
   |   теперь есть объект ответа resp
   |
   |   далее await resp.text():
   |   aiohttp читает данные из сокета (уже в буфере ядра),
   |   и начниает ими пользоваться!



In [52]:
import requests

with requests.Session() as session:
    with session.get('http://ya.ru') as resp:
        text = resp.text
        print(text[:70], '...')

<!doctype html><html prefix="og: http://ogp.me/ns#" lang="ru"><meta ht ...


## Запуск синхронного кода

In [53]:
def blocking_io() -> None:
    print(f"{time.strftime('%X')} start blocking IO")
    time.sleep(5)  # blocks, but at least releases GIL
    # Non-releasing GIL code will block the event loop!
    print(f"{time.strftime('%X')} finished blocking IO")

In [54]:
non_blocking_io = asyncio.to_thread(blocking_io)  # N threads is more expensive than 1 thread with N coros
# await non_blocking_io  <-- blocks if GIL is not released inside the sync coro

In [55]:
print(f"{time.strftime('%X')} start gather")
await asyncio.gather(non_blocking_io, asyncio.sleep(5))
print(f"{time.strftime('%X')} finished gather")

14:48:32 start gather
14:48:32 start blocking IO
14:48:37 finished blocking IO
14:48:37 finished gather


## Debugging asyncio

### Python 3.14: introspection

1. Check introspection.py file!

2. python introspection.py

3. ps aux | grep introspection.py

4. python -m asyncio ps $pid

3 and 4 in one command!

sudo /home/madfyre/.pyenv/versions/3.14.0b2/envs/python_course/bin/python \
  -m asyncio pstree $(ps aux | grep introspection.py | grep -v grep | head -n1 | awk '{print $2}')

└── (T) Task-1
    └──  main /home/madfyre/Documents/YSDA/Python_prep/python_course/private-2025/12.1.Asynchrony/lecture/introspection.py:20
        └──  TaskGroup.__aexit__ /home/madfyre/.pyenv/versions/3.14.0b2/lib/python3.14/asyncio/taskgroups.py:72
            └──  TaskGroup._aexit /home/madfyre/.pyenv/versions/3.14.0b2/lib/python3.14/asyncio/taskgroups.py:121
                ├── (T) Sundowning
                │   └──  play_album /home/madfyre/Documents/YSDA/Python_prep/python_course/private-2025/12.1.Asynchrony/lecture/introspection.py:11
                │       └──  TaskGroup.__aexit__ /home/madfyre/.pyenv/versions/3.14.0b2/lib/python3.14/asyncio/taskgroups.py:72
                │           └──  TaskGroup._aexit /home/madfyre/.pyenv/versions/3.14.0b2/lib/python3.14/asyncio/taskgroups.py:121
                │               ├── (T) TNDNBTG
                │               └── (T) Levitate
                ├── (T) TMBTE
                │   └──  play_album /home/madfyre/Documents/YSDA/Python_prep/python_course/private-2025/12.1.Asynchrony/lecture/introspection.py:11
                │       └──  TaskGroup.__aexit__ /home/madfyre/.pyenv/versions/3.14.0b2/lib/python3.14/asyncio/taskgroups.py:72
                │           └──  TaskGroup._aexit /home/madfyre/.pyenv/versions/3.14.0b2/lib/python3.14/asyncio/taskgroups.py:121
                │               ├── (T) DYWTYLM
                │               └── (T) Aqua Regia
                ├── (T) M83
                │   └──  play_album /home/madfyre/Documents/YSDA/Python_prep/python_course/private-2025/12.1.Asynchrony/lecture/introspection.py:11
                │       └──  TaskGroup.__aexit__ /home/madfyre/.pyenv/versions/3.14.0b2/lib/python3.14/asyncio/taskgroups.py:72
                │           └──  TaskGroup._aexit /home/madfyre/.pyenv/versions/3.14.0b2/lib/python3.14/asyncio/taskgroups.py:121
                │               └── (T) Hurry Up! We are dreaming!
                ├── (T) Heronwater
                │   └──  play_album /home/madfyre/Documents/YSDA/Python_prep/python_course/private-2025/12.1.Asynchrony/lecture/introspection.py:11
                │       └──  TaskGroup.__aexit__ /home/madfyre/.pyenv/versions/3.14.0b2/lib/python3.14/asyncio/taskgroups.py:72
                │           └──  TaskGroup._aexit /home/madfyre/.pyenv/versions/3.14.0b2/lib/python3.14/asyncio/taskgroups.py:121
                │               └── (T) 1000 BARS
                └── (T) FriendlyThug52NGG
                    └──  play_album /home/madfyre/Documents/YSDA/Python_prep/python_course/private-2025/12.1.Asynchrony/lecture/introspection.py:11
                        └──  TaskGroup.__aexit__ /home/madfyre/.pyenv/versions/3.14.0b2/lib/python3.14/asyncio/taskgroups.py:72
                            └──  TaskGroup._aexit /home/madfyre/.pyenv/versions/3.14.0b2/lib/python3.14/asyncio/taskgroups.py:121
                                ├── (T) Graf Monte Cristo
                                └── (T) Cruiser Aurora


`$ PYTHONASYNCIODEBUG=1 python asyncio_program_to_debug.py`

Documentiation: https://docs.python.org/3/library/asyncio-dev.html

How it actually works: https://github.com/python/cpython/tree/main/Lib/asyncio

<center>
<img 
src="https://foni.papik.pro/uploads/posts/2024-10/foni-papik-pro-155x-p-kartinki-yeralash-vse-na-prozrachnom-fone-9.png" alt="sync-vs-async" width=500/>
</center>